Getting started with API tracing
================================

`ReProspect` spans three complementary forms of analysis of CUDA applications: API tracing, kernel profiling, and binary analysis.
Here, we show how to get started with API tracing.

Tracing is the process of collecting and examining information about activities that happen during program execution.
In particular, CUDA API tracing records the CUDA API calls issued by a program: their timing and their correlation with the GPU activities they launch, such as kernel executions.
NVIDIA provides Nsight Systems for this purpose.
`ReProspect` enables a fully programmatic use of this tool: it launches it, collects its output into Python data structures, and supports analyses of these data that can go up to test assertions.

As an example, we consider a program that launches four kernels with a diamond dependency structure.
We programmatically trace its execution, examine the recorded CUDA API calls, and verify the concurrency structure.


About this page
---------------

This page is rendered from a Jupyter notebook, executed at documentation build time, located at `docs/source/getting-started/example_api_tracing.ipynb` in the repository.

To run the example yourself, {download}`download the notebook <example_api_tracing.ipynb>` and open it in JupyterLab.
Alternatively, copy-paste the code snippets successively into an interactive Python session.

The requirements are Python 3.10 or newer and `ReProspect`:

```bash
python -m pip install reprospect
```

JupyterLab may be installed as:

```bash
python -m pip install jupyterlab
```

The example also requires:
- a CUDA Toolkit installation providing `nvcc`;
- an [Nsight Systems CLI installation](https://docs.nvidia.com/nsight-systems/InstallationGuide/index.html#installation-guide) providing the command-line tool `nsys`;
- an [NVTX installation](https://github.com/NVIDIA/NVTX#how-do-i-get-nvtx) providing the `C++` header `nvtx3/nvtx3.hpp`;
- a `C++20`-capable toolchain.

It should be noted that `ReProspect` provides the script {py:mod}`reprospect.utils.installers.nsight_systems` for installing Nsight Systems through `apt`.

Because tracing executes the program, a GPU is also required.

Source code
-----------

Let us consider a program that launches four kernels with a diamond dependency structure:

```{tikz} Four kernels with a diamond dependency structure
:align: center

\begin{tikzpicture}[
  kernel/.style={
    draw,
    rounded corners,
    minimum width=1.2cm,
    minimum height=0.7cm
  },
  >=stealth
]
\node[kernel] (A) at (0, 2)  {kernel A};
\node[kernel] (B) at (-2, 0) {kernel B};
\node[kernel] (C) at (2, 0)  {kernel C};
\node[kernel] (D) at (0, -2) {kernel D};

\draw[->]         (A) -- (B);
\draw[->, dashed] (A) -- node[right=2pt] {\scriptsize event\_fork} (C);
\draw[->]         (B) -- (D);
\draw[->, dashed] (C) -- node[right=2pt] {\scriptsize event\_join} (D);
\end{tikzpicture}
```

The four kernels are launched across two CUDA streams: kernels A, B, and D on one stream, and kernel C on the other, so that kernels B and C may potentially execute concurrently.
Two of the dependencies are thus implied by stream order: A -> B and B -> D.
The two cross-stream dependencies, A -> C and C -> D, are expressed with CUDA events, recorded on one stream and waited for on the other.

The source code features NVTX annotations, for two purposes.
On the one hand, the outer NVTX range `start_end_range` controls when Nsight Systems starts and stops collecting data.
On the other hand, the inner ranges `diamond_range` and `check_range` allow the analyses of the recorded activity to be scoped to these specific regions, as we will see below.
The ranges belong to a dedicated NVTX domain, which keeps them separate from other NVTX annotations, such as those which libraries used by an application may emit.

In [ ]:
CODE = """\
#include <cassert>
#include <source_location>
#include <sstream>
#include <stdexcept>

#include <cuda_runtime.h>
#include <nvtx3/nvtx3.hpp>

inline void check_cudart_call(
    const cudaError_t status,
    const char* const statement,
    const std::source_location& loc = std::source_location::current()) {
    if (status != cudaSuccess) {
        std::ostringstream oss;
        oss << statement << " failed: " << status << " (" << cudaGetErrorName(status)
            << "): " << cudaGetErrorString(status) << " (" << loc.file_name() << ":" << loc.line() << ")";

        throw std::runtime_error(oss.str());
    }
}

#define CHECK_CUDART_CALL(statement) check_cudart_call((statement), #statement)

__global__ void check_add_kernel(int* data, int prev, int val) {
    assert(*data == prev);
    *data += val;
}

__global__ void increment_kernel(int* data) {
    atomicAdd(data, 1);
}

struct ExampleApiTracingDomain {
    static constexpr char const * name{"example_api_tracing_domain"};
};

int main() {
    cudaStream_t stream_0, stream_1;
    CHECK_CUDART_CALL(cudaStreamCreate(&stream_0));
    CHECK_CUDART_CALL(cudaStreamCreate(&stream_1));

    cudaEvent_t event_fork, event_join;
    CHECK_CUDART_CALL(cudaEventCreateWithFlags(&event_fork, cudaEventDisableTiming));
    CHECK_CUDART_CALL(cudaEventCreateWithFlags(&event_join, cudaEventDisableTiming));

    int* data;
    CHECK_CUDART_CALL(cudaMallocAsync(&data, sizeof(int), stream_0));
    CHECK_CUDART_CALL(cudaMemsetAsync(data, 0, sizeof(int), stream_0));

    const auto start_end_range = nvtx3::start_range_in<ExampleApiTracingDomain>("start_end_range");

    {
        nvtx3::scoped_range_in<ExampleApiTracingDomain> diamond_range("diamond_range");
        check_add_kernel<<<1, 1, 0, stream_0>>>(data, 0, 4); // kernel A

        CHECK_CUDART_CALL(cudaEventRecord(event_fork, stream_0));

        increment_kernel<<<1, 1, 0, stream_0>>>(data); // kernel B

        CHECK_CUDART_CALL(cudaStreamWaitEvent(stream_1, event_fork));
        increment_kernel<<<1, 1, 0, stream_1>>>(data); // kernel C

        CHECK_CUDART_CALL(cudaEventRecord(event_join, stream_1));
        CHECK_CUDART_CALL(cudaStreamWaitEvent(stream_0, event_join));

        check_add_kernel<<<1, 1, 0, stream_0>>>(data, 6, 3); // kernel D
    }

    {
        nvtx3::scoped_range_in<ExampleApiTracingDomain> check_range("check_range");
        int data_h;

        CHECK_CUDART_CALL(cudaMemcpyAsync(&data_h, data, sizeof(int), cudaMemcpyDeviceToHost, stream_0));
        CHECK_CUDART_CALL(cudaStreamSynchronize(stream_0));

        if (data_h != 9) {
            throw std::runtime_error("wrong value");
        }
    }

    nvtx3::end_range_in<ExampleApiTracingDomain>(start_end_range);

    CHECK_CUDART_CALL(cudaFreeAsync(data, stream_0));

    CHECK_CUDART_CALL(cudaEventDestroy(event_fork));
    CHECK_CUDART_CALL(cudaEventDestroy(event_join));

    CHECK_CUDART_CALL(cudaStreamDestroy(stream_0));
    CHECK_CUDART_CALL(cudaStreamDestroy(stream_1));
}
"""

Compilation
-----------

Because this example will execute the program, we compile the source code for the native architecture.

In [ ]:
import pathlib
import subprocess

from reprospect.utils import rich_helpers
from reprospect.utils.detect import GPUDetector

print(subprocess.check_output(('nvcc', '--version')).decode().strip())

print(subprocess.check_output(('nsys', '--version')).decode().strip())

visible_gpus = GPUDetector().detect()
print(f'Visible GPUs:\n{rich_helpers.to_string(rich_helpers.df_to_table(visible_gpus))}')

workdir = pathlib.Path.cwd() / 'example_api_tracing'
workdir.mkdir(exist_ok=True)

source = workdir / 'concurrency.cu'
executable = workdir / 'concurrency'

source.write_text(CODE)
_ = subprocess.check_call(('nvcc', '-arch=native', '-std=c++20', '-O3', '-o', executable, source))

Here, {py:class}`reprospect.utils.detect.GPUDetector` is a utility class to detect the visible GPUs through an `nvidia-smi` query.
It returns the data collected about the visible GPUs as a {py:class}`pandas.DataFrame`.
The module {py:mod}`reprospect.utils.rich_helpers` provides functionality for rich rendering of such data frames.

Running API tracing
-------------------

Nsight Systems provides the command-line tool `nsys` for collecting data from the execution of a program. 
Here, we invoke it through the `ReProspect` class {py:class}`reprospect.tools.nsys.session.Session`.
The argument `executable` designates the executable on which to collect data.
The argument `nvtx_capture` is the NVTX range that controls when data collection starts and stops.
The value `'start_end_range@example_api_tracing_domain'` follows the [`nsys` convention](https://docs.nvidia.com/nsight-systems/UserGuide/index.html#cli-profile-command-switch-options) `<range>@<domain>`.
The argument `output` determines the output file; for the passed value `workdir / executable.name`, the report is written to `workdir / f'{executable.name}.nsys-rep'`.

In [ ]:
from reprospect.tools.nsys import Command, Session

ns = Session(
    command=Command(
        executable=executable,
        output=workdir / executable.name,
        nvtx_capture='start_end_range@example_api_tracing_domain',
    ),
)

ns.run(cwd=workdir)

CUDA API Trace default report
-----------------------------

Nsight Systems provides its own functionality for post-processing the collected data into summary and trace reports, as described [here](https://docs.nvidia.com/nsight-systems/AnalysisGuide/index.html#statistical-reports-shipped-with-product-name).
Nsight Systems generates these default reports through SQL queries on an SQLite export of the output file.

The `ReProspect` method {py:meth}`reprospect.tools.nsys.session.Session.extract_statistical_report` allows such reports to be retrieved as a {py:class}`pandas.DataFrame`.
Here, we retrieve the report named `cuda_api_trace`, which contains a trace of the CUDA API calls, with their start times and durations.

We first export the `.nsys-rep` report to an SQLite database.
The method {py:meth}`~reprospect.tools.nsys.session.Session.extract_statistical_report` then invokes `nsys stats` on this database.
The same database will be queried directly in the sections below.

In [ ]:
sqlite_database = ns.export_to_sqlite(cwd=workdir)
print(f'SQLite database: {sqlite_database}')

In [ ]:
cuda_api_trace_report = ns.extract_statistical_report(report='cuda_api_trace')
print(f'Nsight Systems cuda_api_trace report:\n{rich_helpers.to_string(rich_helpers.df_to_table(cuda_api_trace_report))}')

Querying CUDA API tracing data
------------------------------

Beyond the default reports of the previous section, the Nsight Systems documentation recommends the SQLite export of the `.nsys-rep` file as the primary means for custom analyses of the collected data, with full access to all recorded details and their correlations.
The Nsight Systems documentation provides the SQLite schema reference [here](https://docs.nvidia.com/nsight-systems/AnalysisGuide/index.html#sqlite-schema-reference).

The `ReProspect` class {py:class}`reprospect.tools.nsys.report.Report` provides access to the database.
The class is used as a context manager, which opens and closes the connection to the database.

The property {py:attr}`reprospect.tools.nsys.report.Report.tables` provides a list of the names of the tables available in the database.

In [ ]:
from reprospect.tools.nsys import Report

report = Report(db=ns.command.output.with_suffix('.sqlite'))

with report:
    tables = report.tables
    print(f'Tables:\n{tables}')

The method {py:meth}`reprospect.tools.nsys.report.Report.table` retrieves a table as a {py:class}`pandas.DataFrame`.
Let us retrieve the tables:
- `CUPTI_ACTIVITY_KIND_RUNTIME` with the trace of the CUDA runtime API calls;
- `CUPTI_ACTIVITY_KIND_CUDA_EVENT` with details relevant to the CUDA events;
- `CUPTI_ACTIVITY_KIND_SYNCHRONIZATION` with details relevant to the CUDA synchronization operations;
- `CUPTI_ACTIVITY_KIND_KERNEL` with details relevant to the kernel executions;
- `ENUM_NSYS_EVENT_CLASS`, which we will use below to interpret the `eventClass` column of the runtime table;
- `StringIds`, which relates string identifiers to the strings themselves.

Here, `CUPTI` stands for the [CUDA Profiling Tools Interface (CUPTI)](https://developer.nvidia.com/cupti), which Nsight Systems uses to collect the associated data.

In [ ]:
with report:
    runtime = report.table(name='CUPTI_ACTIVITY_KIND_RUNTIME')
    print(f'Table CUPTI_ACTIVITY_KIND_RUNTIME:\n{rich_helpers.to_string(rich_helpers.df_to_table(runtime))}')

    cuda_event = report.table(name='CUPTI_ACTIVITY_KIND_CUDA_EVENT')
    print(f'Table CUPTI_ACTIVITY_KIND_CUDA_EVENT:\n{rich_helpers.to_string(rich_helpers.df_to_table(cuda_event))}')

    synchronization = report.table(name='CUPTI_ACTIVITY_KIND_SYNCHRONIZATION')
    print(f'Table CUPTI_ACTIVITY_KIND_SYNCHRONIZATION:\n{rich_helpers.to_string(rich_helpers.df_to_table(synchronization))}')

    kernel = report.table(name='CUPTI_ACTIVITY_KIND_KERNEL')
    print(f'Table CUPTI_ACTIVITY_KIND_KERNEL, for brevity only its column names:\n{kernel.columns}')

    enum_event_class = report.table(name='ENUM_NSYS_EVENT_CLASS')
    print(f'Table ENUM_NSYS_EVENT_CLASS first two rows:\n{rich_helpers.to_string(rich_helpers.df_to_table(enum_event_class[:2]))}')

    stringids = report.table(name='StringIds')
    first_runtime_row = runtime.iloc[0]
    print(f'Table StringIds row related to first runtime table row:\n{rich_helpers.to_string(rich_helpers.df_to_table(stringids[stringids["id"] == first_runtime_row["nameId"]]))}')

As we can observe, analysing the data requires exploiting the relationships between the tables.
Indeed, to retrieve the name of the activity described in the first row of the runtime table, we must read the `nameId` from the runtime table and correlate it via the `id` in the `StringIds` table with its corresponding `value`.

Querying CUDA API tracing data by nested NVTX range
---------------------------------------------------

`ReProspect` provides functionalities to facilitate lookups in the SQLite database.
The method {py:meth}`reprospect.tools.nsys.report.Report.get_events` provides focused lookups by nested NVTX range, with automatic correlation of string identifiers.
For tables featuring `start` and `end` columns, it selects the rows whose time span is within a given nested NVTX range.
The argument `accessors` designates the path of nested NVTX ranges, from the outermost inward.

In [ ]:
with report:
    print(report.nvtx_events)

In [ ]:
with report:
    diamond_runtime = report.get_events(table='CUPTI_ACTIVITY_KIND_RUNTIME', accessors=['start_end_range', 'diamond_range'])
    print(f'Runtime events in diamond range:\n{diamond_runtime}')

Some runtime activity names carry a suffix such as `_v7000` and `_v3020`.
The function {py:func}`reprospect.tools.nsys.report.strip_cuda_api_suffix` allows such suffixes to be stripped, so that the analysis does not depend on them.
The runtime table may also contain CUDA driver activity, such as `cuLibraryLoadData`, `cuLibraryGetKernel`, `cuKernelGetName` related to module loading and name resolution.
Such entries may depend on the execution environment rather than on the program; we exclude them by selecting only the rows whose `eventClass` corresponds to the label `'CUDA runtime'` in the `ENUM_NSYS_EVENT_CLASS` table.

In [ ]:
from reprospect.tools.nsys import strip_cuda_api_suffix

RUNTIME_EVENT_CLASS_ID = Report.single_row(data=enum_event_class[enum_event_class['label'] == 'CUDA runtime'])['id']

diamond_cuda_runtime = (
    diamond_runtime.loc[diamond_runtime['eventClass'] == RUNTIME_EVENT_CLASS_ID]
    .assign(api=lambda df: df['name'].map(strip_cuda_api_suffix))
)
print(f'CUDA runtime api calls in diamond range:\n{diamond_cuda_runtime["api"].to_list()}')

The method {py:meth}`reprospect.tools.nsys.report.Report.get_correlated_row` facilitates correlated lookups across tables in the database.
By default, it uses the `correlationId` column to join both tables.
The columns to use for the correlation can also be designated explicitly, as needed for instance when retrieving the name of a kernel by correlating its `demangledName` identifier in the `CUPTI_ACTIVITY_KIND_KERNEL` table with the `id` in the `StringIds` table:

In [ ]:
kernel_a = Report.get_correlated_row(src=diamond_cuda_runtime[diamond_cuda_runtime['api'] == 'cudaLaunchKernel'].iloc[0], dst=kernel)
print('StreamId for kernel A:', kernel_a['streamId'])

kernel_a_demangled_name = Report.get_correlated_row(src=kernel_a, dst=stringids, correlation_src='demangledName', correlation_dst='id')
print(f'Demangled name for kernel A: {kernel_a_demangled_name["value"]}')

Assertions on CUDA API tracing data
-----------------------------------

With the collected data gathered in Python data structures, the analysis can now go all the way to test assertions.
Here, we programmatically verify the concurrency structure.

First, we verify the sequence of the CUDA runtime API calls.
The sequence assertion checks for exact equality: after stripping the name suffixes and excluding the driver activity, the remaining sequence is fully determined by the order of the host-side calls in the source code.

In [ ]:
assert diamond_cuda_runtime['api'].to_list() == [
    'cudaLaunchKernel',
    'cudaEventRecord',
    'cudaLaunchKernel',
    'cudaStreamWaitEvent',
    'cudaLaunchKernel',
    'cudaEventRecord',
    'cudaStreamWaitEvent',
    'cudaLaunchKernel',
]

Next, we verify the streams the kernels are launched on.
In particular, we check that the potentially concurrent kernels B and C are launched on different streams and that kernels A, B, and D are launched on the same stream.

In [ ]:
launches = diamond_cuda_runtime[diamond_cuda_runtime['api'] == 'cudaLaunchKernel']
kernel_a, kernel_b, kernel_c, kernel_d = (Report.get_correlated_row(src=launches.iloc[idx], dst=kernel) for idx in range(4))
assert kernel_b['streamId'] != kernel_c['streamId']

assert kernel_a['streamId'] == kernel_b['streamId'] == kernel_d['streamId']

Finally, we verify the dependencies that form the diamond.
In particular, for each dependency, we check that the event is recorded on the predecessor's stream, that the wait executes on the successor's stream, and that record and wait reference the same event.

In [ ]:
event_records = diamond_cuda_runtime[diamond_cuda_runtime['api'] == 'cudaEventRecord']
stream_wait_events = diamond_cuda_runtime[diamond_cuda_runtime['api'] == 'cudaStreamWaitEvent']

event_record_fork = Report.get_correlated_row(src=event_records.iloc[0], dst=cuda_event)
stream_wait_event_fork = Report.get_correlated_row(src=stream_wait_events.iloc[0], dst=synchronization)
assert event_record_fork['streamId'] == kernel_a['streamId']
assert stream_wait_event_fork['streamId'] == kernel_c['streamId']
assert stream_wait_event_fork['eventId'] == event_record_fork['eventId']

event_record_join = Report.get_correlated_row(src=event_records.iloc[1], dst=cuda_event)
stream_wait_event_join = Report.get_correlated_row(src=stream_wait_events.iloc[1], dst=synchronization)
assert event_record_join['streamId'] == kernel_c['streamId']
assert stream_wait_event_join['streamId'] == kernel_d['streamId']
assert stream_wait_event_join['eventId'] == event_record_join['eventId']

Outlook
-------

In the example of this notebook, the traced sequence can be read off the source code directly.
One context in which the proposed approach becomes most valuable is that of abstraction layers.
Portability libraries such as `Kokkos` let applications express computations at a higher level of abstraction and map them to CUDA API calls internally.
Likewise, the `C++26` `std::execution` model, whose customization for CUDA is under development in [NVIDIA's `CCCL` library](https://github.com/NVIDIA/cccl), lowers declarative descriptions of asynchronous work, including fork–join structures like the diamond of this notebook, to streams and events.
API tracing then allows verifying that these mappings produce the intended calls.
With `ReProspect`'s fully programmatic approach, such verifications can run as tests in CI/CD pipelines.
API tracing is used in our {ref}`Kokkos View allocation case study <example-kokkos-view-allocation>`, where it elucidates benchmarking results by identifying the CUDA API calls that `Kokkos` issues when allocating a `Kokkos::View` under different scenarios.